In [1]:
# Install required libraries
!pip install pandas scikit-learn -q

In [2]:
# Import required libraries
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
# Load the dataset (upload ecommerce_orders.csv to the Colab session first)
data = pd.read_csv("/content/Dataset for Data Analytics - Sheet1.csv")

In [4]:
# Display the dataset
data

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,ORD201195,2024-06-20,C21126,Desk,1,107.04,392 Main St,Credit Card,Cancelled,TRK38009181,6,FREESHIP,Google,107.04
1196,ORD201196,2024-03-04,C20095,Monitor,2,662.53,778 Main St,Online,Cancelled,TRK69207593,5,NaN,Facebook,1325.06
1197,ORD201197,2023-07-13,C79674,Tablet,2,436.84,275 Main St,Online,Delivered,TRK88039356,2,FREESHIP,Instagram,873.68
1198,ORD201198,2024-08-22,C64753,Chair,4,262.52,509 Main St,Debit Card,Cancelled,TRK71683331,4,WINTER15,Instagram,1050.08


In [5]:
# Replace missing coupon codes with a placeholder label
data["CouponCode"] = data["CouponCode"].fillna("NoCoupon")

In [6]:
# Combine relevant order attributes into a single text feature per order
data["Profile"] = data["PaymentMethod"] + " " + data["OrderStatus"] + " " + data["ReferralSource"] + " " + data["CouponCode"]

In [7]:
# Create a text corpus by grouping all order attributes into one profile per product
product_profiles = data.groupby("Product")["Profile"].apply(lambda x: " ".join(x)).reset_index()
product_profiles.columns = ["Product", "ProfileText"]

In [8]:
# Display the product profiles
product_profiles

,Product,ProfileText
0,Chair,Debit Card Returned Facebook SAVE10 Cash Pendi...
1,Desk,Credit Card Shipped Google SAVE10 Gift Card Pe...
2,Laptop,Gift Card Returned Facebook SAVE10 Credit Card...
3,Monitor,Debit Card Shipped Instagram SAVE10 Cash Shipp...
4,Phone,Online Shipped Referral SAVE10 Credit Card Shi...
5,Printer,Online Delivered Email SAVE10 Cash Delivered G...
6,Tablet,Credit Card Cancelled Email FREESHIP Credit Ca...


In [9]:
# Take at least three shopping preferences as user input
user_preferences = ["Debit Card", "Instagram", "SAVE10"]

In [10]:
# Convert the product profile text into TF-IDF vectors
corpus = product_profiles["ProfileText"].tolist()
vectorizer = TfidfVectorizer()
product_vectors = vectorizer.fit_transform(corpus)

In [11]:
# Convert the user preferences into a TF-IDF vector using the same vectorizer
user_text = " ".join(user_preferences)
user_vector = vectorizer.transform([user_text])

In [12]:
# Calculate Cosine Similarity between the user vector and every product profile
similarity_scores = cosine_similarity(user_vector, product_vectors)

In [13]:
# Store similarity scores alongside each product
product_profiles["Similarity Score"] = similarity_scores[0]

In [14]:
# Sort similarity scores in descending order
ranked_products = product_profiles.sort_values(by="Similarity Score", ascending=False)

In [15]:
# Filter and return only the Top 3 recommendations
top_3_recommendations = ranked_products.head(3)

In [16]:
# Display the user's selected shopping preferences
print("User Preferences:")
for preference in user_preferences:
    print("-", preference)

User Preferences:
- Debit Card
- Instagram
- SAVE10


In [17]:
# Display Top 3 recommended products with product name and similarity score
print("Top 3 Recommended Products:\n")
for index, row in top_3_recommendations.iterrows():
    print("Product:", row["Product"])
    print("Similarity Score:", round(row["Similarity Score"], 2))
    print()

Top 3 Recommended Products:

Product: Laptop
Similarity Score: 0.57

Product: Desk
Similarity Score: 0.57

Product: Phone
Similarity Score: 0.57

